# Hướng 3: reranker mạnh hơn (Qwen2-VL-7B và LoRA cho Qwen2-VL-2B)

Hướng 1 đã rerank top-50 của **MagicLens tinh chỉnh** bằng Qwen2-VL-2B. Hướng 3 giữ nguyên tập ứng viên
đó và **chỉ thay VLM**, để biết một reranker mạnh hơn giúp thêm được bao nhiêu. Hướng dẫn đầy đủ:
`docs/huong3_reranker_manh_hon.md`.

| Phần | Việc | Thời gian ước tính | Ghi ra Drive |
|---|---|---|---|
| A | Chấm lại bản 2B ở bfloat16 trên 200 truy vấn, so với điểm float16 của Hướng 1 | khoảng 10 phút | `vlm_scores_2b_bf16_check.jsonl` |
| B | Chấm 93 050 cặp bằng Qwen2-VL-7B (zero-shot) | 3–4 giờ trên L4, ít hơn trên A100 | `vlm_scores_7b.jsonl` |
| C | Huấn luyện LoRA cho Qwen2-VL-2B trên tập train FashionIQ, chọn checkpoint trên dev | khoảng 2–2.5 giờ | `lora_2b/` |
| D | Chấm 93 050 cặp bằng 2B + checkpoint LoRA tốt nhất | khoảng 1 giờ | `vlm_scores_2b_lora.jsonl` |

Các phần độc lập với nhau, trừ D cần C xong trước. Có thể chạy mỗi phần trong một phiên Colab khác nhau.
Mọi phần đều **chạy tiếp được** sau khi Colab bị ngắt: kết nối lại, chạy lại ô 1–5 rồi chạy lại ô của phần đó.

**Trước khi chạy:**
1. Trên Drive, thư mục **`MyDrive/magiclens_vlm_ft/`** của Hướng 1 phải còn `bundle.zip` và
   `vlm_scores.jsonl` (cùng `vlm_scores.jsonl.meta.json`). Không cần tải lại.
2. Riêng phần C và D: tải `vlm_lora/lora_train_bundle.zip` (khoảng 850 MB, tạo bằng `vlm_lora_prepare.py`)
   vào **cùng thư mục đó**, không giải nén.
3. **Runtime → Change runtime type**: chọn **A100** (nhanh nhất) hoặc **L4**. **T4 không dùng được**: không có
   bfloat16, và bản 7B (khoảng 17 GB) không vừa 15 GB bộ nhớ của T4.

**Mỗi khi Colab khởi động lại**, mọi biến, thư viện vừa cài và dữ liệu đã giải nén đều mất; chỉ dữ liệu trên
Drive còn lại. Vì vậy luôn chạy lại ô 1–5 trước. Colab tính phí theo thời gian GPU được kết nối, kể cả khi
không chạy gì: xong việc thì **Runtime → Disconnect and delete runtime**.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Chưa có GPU: Runtime → Change runtime type → A100 hoặc L4.")
print("GPU:", torch.cuda.get_device_name(), "| bfloat16:", torch.cuda.is_bf16_supported())
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("GPU này không có bfloat16 (ví dụ T4): đổi sang A100 hoặc L4.")

## 2. Kết nối Google Drive
Colab sẽ hỏi quyền truy cập Drive; chọn tài khoản chứa thư mục `magiclens_vlm_ft`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Cấu hình

Thường không cần sửa gì. Các trường hợp cần đổi (sửa rồi chạy lại ô này):
- Phần B báo **CUDA out of memory**: giảm `BATCH_7B` (ví dụ 4). Trên A100 có thể tăng lên 25 cho nhanh hơn.
  Đổi batch giữa chừng không sao, kết quả không đổi.
- Muốn phần B nhanh hơn: `TOP_K_7B = 30` (ít hơn 40% số cặp, nhưng trần R@10 giảm từ 57.9 xuống 50.7).
  Phải quyết định **trước khi** bắt đầu phần B; đổi giữa chừng thì script sẽ từ chối chạy tiếp.

In [ ]:
import codecs, json, os, subprocess, sys

DRIVE_DIR = "/content/drive/MyDrive/magiclens_vlm_ft"  # thư mục của Hướng 1 trên Drive
BUNDLE_DIR = "/content/bundle"            # ứng viên val của Hướng 1 (bundle.zip)
LORA_TRAIN_DIR = "/content/lora_train"    # dữ liệu train của LoRA (lora_train_bundle.zip)
MODEL_2B = "Qwen/Qwen2-VL-2B-Instruct"
MODEL_7B = "Qwen/Qwen2-VL-7B-Instruct"

OUT_FP16 = f"{DRIVE_DIR}/vlm_scores.jsonl"                # điểm 2B float16 của Hướng 1 (đã có)
OUT_BF16 = f"{DRIVE_DIR}/vlm_scores_2b_bf16_check.jsonl"  # phần A
N_CHECK = 200                                             # số truy vấn của phần A
OUT_7B = f"{DRIVE_DIR}/vlm_scores_7b.jsonl"               # phần B
TOP_K_7B = 50
BATCH_7B = 10
LORA_DIR = f"{DRIVE_DIR}/lora_2b"                         # phần C: checkpoint và nhật ký
OUT_LORA = f"{DRIVE_DIR}/vlm_scores_2b_lora.jsonl"        # phần D
BATCH_2B = 10


def sh(cmd):
    # Chạy một lệnh shell, in kết quả ngay khi có; báo lỗi nếu lệnh thất bại.
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    dec = codecs.getincrementaldecoder("utf-8")(errors="replace")
    while True:
        chunk = p.stdout.read1(65536)
        if not chunk:
            break
        sys.stdout.write(dec.decode(chunk))
        sys.stdout.flush()
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"Lệnh bị lỗi (mã {rc}). Đọc thông báo ngay phía trên để biết nguyên nhân.")


assert os.path.exists(f"{DRIVE_DIR}/bundle.zip"), (
    f"Không thấy {DRIVE_DIR}/bundle.zip của Hướng 1: chạy ô 2 (kết nối Drive) và kiểm tra thư mục.")
print("OK. Có lora_train_bundle.zip (cho phần C, D):",
      os.path.exists(f"{DRIVE_DIR}/lora_train_bundle.zip"))

## 4. Cài thư viện

Colab đã có sẵn PyTorch. Ô này cài đúng các bản đã kiểm thử: transformers 4.57.6 (như Hướng 1) và peft 0.17.1
(cho LoRA). Pip có thể in `ERROR: pip's dependency resolver ...`: đó chỉ là cảnh báo xung đột phiên bản,
**bỏ qua được** nếu dòng cuối hiện đúng hai phiên bản trên.

In [ ]:
!pip install -q "transformers==4.57.6" "peft==0.17.1" accelerate
import transformers, peft
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)

## 5. Giải nén dữ liệu

Giải nén `bundle.zip` của Hướng 1 (ứng viên val) và, nếu có, `lora_train_bundle.zip`. Các script trong gói
LoRA là bản mới nhất (có `--adapter`), nên được dùng cho mọi phần khi có gói này.

In [ ]:
if "OUT_7B" not in globals():
    raise RuntimeError("Phiên Colab mới chưa có cấu hình (Colab đã khởi động lại): "
                       "chạy lại ô 1, 2, 3, 4, 5 rồi chạy lại ô này.")

sh(f'mkdir -p {BUNDLE_DIR} && unzip -q -o "{DRIVE_DIR}/bundle.zip" -d {BUNDLE_DIR}')
cand = json.load(open(f"{BUNDLE_DIR}/candidates.json"))
print("Ứng viên của bước 1:", cand.get("ranker"), f"| {len(cand['queries'])} truy vấn × top-{cand['top_k']}")
if cand.get("ranker") != "magic_lens_clip_base_fiq_dress_ft.pkl":
    raise RuntimeError("bundle.zip này không phải của Hướng 1 (vlm_rerank_ft): kiểm tra DRIVE_DIR.")
HAS_LORA = os.path.exists(f"{DRIVE_DIR}/lora_train_bundle.zip")
if HAS_LORA:
    sh(f'mkdir -p {LORA_TRAIN_DIR} && unzip -q -o "{DRIVE_DIR}/lora_train_bundle.zip" -d {LORA_TRAIN_DIR}')
    print(len(os.listdir(f"{LORA_TRAIN_DIR}/images")), "ảnh train")
SCRIPTS = LORA_TRAIN_DIR if HAS_LORA else BUNDLE_DIR
print("Script dùng từ:", SCRIPTS, "| phần C, D:", "sẵn sàng" if HAS_LORA else
      "chưa có lora_train_bundle.zip trên Drive (chạy được phần A, B)")

## Phần A. Kiểm tra kiểu số (khoảng 10 phút)

Bản 7B chạy ở **bfloat16** (miền giá trị rộng hơn float16, ít bị tràn số), còn bản 2B của Hướng 1 chạy ở
**float16**. Ô này chấm lại bản 2B ở bfloat16 trên `N_CHECK` truy vấn đầu và so với điểm float16 đã có. Nếu hai
bộ điểm gần như trùng nhau thì chênh lệch giữa 2B và 7B ở phần B đến từ kích thước mô hình, không phải từ kiểu số.

In [ ]:
if "OUT_7B" not in globals():
    raise RuntimeError("Phiên Colab mới chưa có cấu hình (Colab đã khởi động lại): "
                       "chạy lại ô 1, 2, 3, 4, 5 rồi chạy lại ô này.")

if not os.path.exists(OUT_FP16):
    raise RuntimeError(f"Không thấy {OUT_FP16}: phần A cần điểm 2B float16 của Hướng 1 trên Drive.")
sh(f'python {SCRIPTS}/vlm_rerank.py --bundle_dir {BUNDLE_DIR} --out "{OUT_BF16}" --model {MODEL_2B} '
   f'--dtype bfloat16 --limit {N_CHECK} --batch_size {BATCH_2B}')
import numpy as np


def read_scores(path):
    out = {}
    for line in open(path):
        try:
            r = json.loads(line)
            out[r["q"]] = np.array(r["scores"])
        except (json.JSONDecodeError, KeyError):
            pass
    return out


a, b = read_scores(OUT_FP16), read_scores(OUT_BF16)
qs = sorted(set(a) & set(b))
A, B = np.stack([a[q] for q in qs]), np.stack([b[q] for q in qs])
rank = lambda x: np.argsort(np.argsort(-x, axis=1), axis=1)
pearson = np.corrcoef(A.ravel(), B.ravel())[0, 1]
spearman = np.mean([np.corrcoef(x, y)[0, 1] for x, y in zip(rank(A), rank(B))])
print(f"{len(qs)} truy vấn: Pearson {pearson:.4f}, lệch trung bình {np.abs(A - B).mean():.3f} "
      f"(độ lệch chuẩn của điểm {A.std():.2f}), Spearman trong truy vấn {spearman:.4f}")
print("KẾT LUẬN:", "bfloat16 gần như không đổi điểm." if pearson >= 0.99 and spearman >= 0.98 else
      "bfloat16 làm điểm thay đổi đáng kể: ghi chú điều này khi so 7B với 2B.")

## Phần B. Qwen2-VL-7B zero-shot

Ô B1 chạy thử 3 truy vấn × 20 ứng viên, **hai lần**: một lần mã hóa lại mọi ảnh (cách chuẩn) và một lần dùng
lại đặc trưng ảnh đã tính (cách nhanh, mặc định). Lần đầu ô này tải Qwen2-VL-7B (khoảng 17 GB, vài phút). Ô in ra:
- **OK**: hai cách cho cùng điểm, ô B2 dùng cách nhanh.
- **CẢNH BÁO**: cách nhanh lệch hoặc lỗi; ô B2 tự dùng cách chuẩn (chậm hơn nhưng đúng).

Ô B2 chấm đầy đủ 1861 truy vấn × `TOP_K_7B` ứng viên, ghi thẳng lên Drive sau mỗi truy vấn. Thanh tiến trình
hiện tốc độ và thời gian còn lại. Bị ngắt thì chạy lại ô 1–5 rồi ô B2 (không cần chạy lại B1).

In [ ]:
if "OUT_7B" not in globals():
    raise RuntimeError("Phiên Colab mới chưa có cấu hình (Colab đã khởi động lại): "
                       "chạy lại ô 1, 2, 3, 4, 5 rồi chạy lại ô này.")

# Ô B1: chạy thử
NO_CACHE_MARK = f"{DRIVE_DIR}/use_no_vision_cache_7b"
sh("rm -rf /content/smoke7b")
for name, flag in (("plain", "--no_vision_cache"), ("cache", "")):
    try:
        sh(f"python {SCRIPTS}/vlm_rerank.py --bundle_dir {BUNDLE_DIR} --out /content/smoke7b/{name}.jsonl "
           f"--model {MODEL_7B} --dtype bfloat16 --top_k 20 --limit 3 --batch_size {BATCH_7B} {flag}")
    except RuntimeError as e:
        print("Lỗi:", e)
smoke = lambda n: ([json.loads(l) for l in open(f"/content/smoke7b/{n}.jsonl")]
                   if os.path.exists(f"/content/smoke7b/{n}.jsonl") else [])
plain, cache = smoke("plain"), smoke("cache")
if len(plain) < 3:
    raise RuntimeError("LỖI: lần chạy chuẩn không hoàn thành; đọc thông báo ở trên "
                       "(CUDA out of memory → giảm BATCH_7B ở ô 3).")
diff = (max(abs(x - y) for r, s in zip(plain, cache) for x, y in zip(r["scores"], s["scores"]))
        if len(cache) == 3 else float("inf"))
if diff < 0.3:
    if os.path.exists(NO_CACHE_MARK):
        os.remove(NO_CACHE_MARK)
    print(f"OK: hai cách lệch nhau tối đa {diff:.4f}; ô B2 dùng cách nhanh.")
else:
    open(NO_CACHE_MARK, "w").close()
    print("CẢNH BÁO: cách nhanh " + ("bị lỗi" if diff == float("inf") else f"lệch {diff:.4f}")
          + "; ô B2 sẽ dùng --no_vision_cache.")

In [ ]:
if "OUT_7B" not in globals():
    raise RuntimeError("Phiên Colab mới chưa có cấu hình (Colab đã khởi động lại): "
                       "chạy lại ô 1, 2, 3, 4, 5 rồi chạy lại ô này.")

# Ô B2: chấm đầy đủ bằng 7B (chạy tiếp được)
NO_CACHE_MARK = f"{DRIVE_DIR}/use_no_vision_cache_7b"
if os.path.exists(OUT_7B + ".meta.json"):  # chạy tiếp: giữ đúng cách chạy lúc bắt đầu
    no_cache = not json.load(open(OUT_7B + ".meta.json")).get("vision_cache", True)
else:
    no_cache = os.path.exists(NO_CACHE_MARK)
sh(f'python {SCRIPTS}/vlm_rerank.py --bundle_dir {BUNDLE_DIR} --out "{OUT_7B}" --model {MODEL_7B} '
   f'--dtype bfloat16 --top_k {TOP_K_7B} --batch_size {BATCH_7B}' + (" --no_vision_cache" if no_cache else ""))

## Phần C. Huấn luyện LoRA cho Qwen2-VL-2B

Cần `lora_train_bundle.zip` trên Drive (ô 5 phải báo "phần C, D: sẵn sàng"). Mỗi truy vấn train (5441 truy vấn
dress) cho 6 cặp: ảnh đích (nhãn "Yes"), 4 mẫu âm khó rút từ hạng 6–50 của MagicLens zero-shot (bỏ 5 hạng đầu; rút lại mỗi epoch)
và chính ảnh truy vấn (nhãn "No"). Học 2 epoch (2178 bước, mỗi bước 30 cặp), LoRA r = 16 trên các phép chiếu
attention của phần ngôn ngữ, tốc độ học 1e-4.

Trước khi học và sau mỗi nửa epoch, script chấm 200 truy vấn dev × top-20 ứng viên của Hướng 1 và ghi **AUC dev**
(khả năng tách ảnh đích khỏi các ứng viên khác; 0.5 là ngẫu nhiên). Checkpoint lưu vào `lora_2b/ckpt-…` trên Drive;
cuối cùng `lora_2b/best.json` ghi checkpoint có AUC dev cao nhất. Dòng "step 0" là mô hình chưa học: các
checkpoint phải vượt mốc này thì LoRA mới có tác dụng.

Bị ngắt thì chạy lại ô 1–5 rồi ô này: script học tiếp từ checkpoint gần nhất (mất tối đa nửa epoch).
Báo **CUDA out of memory** thì thêm `--grad_ckpt` vào cuối lệnh (chậm hơn khoảng 30%, kết quả không đổi).

In [ ]:
if "OUT_7B" not in globals():
    raise RuntimeError("Phiên Colab mới chưa có cấu hình (Colab đã khởi động lại): "
                       "chạy lại ô 1, 2, 3, 4, 5 rồi chạy lại ô này.")

if not HAS_LORA:
    raise RuntimeError("Chưa có lora_train_bundle.zip trên Drive: tải lên DRIVE_DIR rồi chạy lại ô 5.")
sh(f'python {LORA_TRAIN_DIR}/vlm_lora_train.py --train_dir {LORA_TRAIN_DIR} --val_dir {BUNDLE_DIR} '
   f'--out_dir "{LORA_DIR}" --model {MODEL_2B}')
best = json.load(open(f"{LORA_DIR}/best.json"))
for c in best["checkpoints"]:
    print(f"bước {c['step']:5d}  {c['ckpt'] or 'chưa học':12s}  AUC dev {c['dev_auc']:.4f}  MRR dev {c['dev_mrr']:.4f}")
print("Checkpoint tốt nhất trên dev:", best["best"])

## Phần D. Chấm điểm bằng checkpoint LoRA tốt nhất (khoảng 1 giờ)

Đọc `lora_2b/best.json`, rồi chấm đúng 93 050 cặp như Hướng 1 bằng 2B + checkpoint đó (bfloat16, như lúc
huấn luyện). Bị ngắt thì chạy lại ô 1–5 rồi ô này.

In [ ]:
if "OUT_7B" not in globals():
    raise RuntimeError("Phiên Colab mới chưa có cấu hình (Colab đã khởi động lại): "
                       "chạy lại ô 1, 2, 3, 4, 5 rồi chạy lại ô này.")

if not HAS_LORA:
    raise RuntimeError("Chưa có lora_train_bundle.zip trên Drive: tải lên DRIVE_DIR rồi chạy lại ô 5.")
if not os.path.exists(f"{LORA_DIR}/best.json"):
    raise RuntimeError("Chưa có lora_2b/best.json: chạy xong phần C trước.")
adapter = f"{LORA_DIR}/{json.load(open(f'{LORA_DIR}/best.json'))['best']}"
print("Adapter:", adapter)
sh(f'python {SCRIPTS}/vlm_rerank.py --bundle_dir {BUNDLE_DIR} --out "{OUT_LORA}" --model {MODEL_2B} '
   f'--dtype bfloat16 --top_k 50 --batch_size {BATCH_2B} --adapter "{adapter}"')

## 6. Kiểm tra tiến độ và tải kết quả về máy

Ô đầu in tiến độ của từng phần. Ô sau tải về các tệp đã có (trình duyệt sẽ hỏi lưu từng tệp); cũng có thể tải
bằng giao diện Google Drive. Trên máy, chép:
- `vlm_scores_2b_bf16_check.jsonl`, `vlm_scores_7b.jsonl`, `vlm_scores_2b_lora.jsonl` **cùng các tệp
  `.meta.json` đi kèm** vào `vlm_rerank_ft/`;
- `best.json` và `train_log.jsonl` của `lora_2b/` vào `vlm_lora/`.

Rồi chạy (bỏ dòng của phần chưa có):

```bash
.venv/bin/python vlm_rerank_eval.py --candidates vlm_rerank_ft/candidates.json \
    --scores vlm_rerank_ft/vlm_scores_7b.jsonl --output ./output/huong3
.venv/bin/python vlm_rerank_eval.py --candidates vlm_rerank_ft/candidates.json \
    --scores vlm_rerank_ft/vlm_scores_2b_lora.jsonl --output ./output/huong3
.venv/bin/python huong3_tong_hop.py
```

In [ ]:
if "OUT_7B" not in globals():
    raise RuntimeError("Phiên Colab mới chưa có cấu hình (Colab đã khởi động lại): "
                       "chạy lại ô 1, 2, 3, 4, 5 rồi chạy lại ô này.")

n_total = len(json.load(open(f"{BUNDLE_DIR}/candidates.json"))["queries"])
for name, path, total in (("A (bf16, 2B)", OUT_BF16, N_CHECK), ("B (7B)", OUT_7B, n_total),
                          ("D (2B + LoRA)", OUT_LORA, n_total)):
    if not os.path.exists(path):
        print(f"{name}: chưa chạy")
        continue
    recs = []
    for line in open(path):
        try:
            recs.append(json.loads(line))
        except json.JSONDecodeError:
            pass
    secs, pairs = sum(r["sec"] for r in recs), sum(len(r["scores"]) for r in recs)
    print(f"{name}: {len(recs)}/{total} truy vấn, {secs / 3600:.2f} giờ ({secs / max(pairs, 1) * 1000:.0f} ms/cặp)"
          + ("  → XONG" if len(recs) >= total else "  → CHƯA XONG: chạy lại ô của phần này"))
print("C (LoRA):", "XONG, tốt nhất " + json.load(open(f"{LORA_DIR}/best.json"))["best"]
      if os.path.exists(f"{LORA_DIR}/best.json") else "chưa xong")

In [ ]:
if "OUT_7B" not in globals():
    raise RuntimeError("Phiên Colab mới chưa có cấu hình (Colab đã khởi động lại): "
                       "chạy lại ô 1, 2, 3, 4, 5 rồi chạy lại ô này.")

from google.colab import files
for path in (OUT_BF16, OUT_7B, OUT_LORA):
    if os.path.exists(path):
        files.download(path)
        files.download(path + ".meta.json")
for name in ("best.json", "train_log.jsonl"):
    if os.path.exists(f"{LORA_DIR}/{name}"):
        files.download(f"{LORA_DIR}/{name}")